# 03. ML Agent — 머신러닝 Tool 통합
**Day 14**

> Day 8~11에서 배운 머신러닝 모델을 **LangChain `@tool`** 로 감싸 Agent가 호출합니다.  
> 구현 코드: `ml_tools/` 폴더 (로지스틱 회귀 · XGBoost · One-Class SVM)

---
## 0. 설치 & 경로

In [1]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-classic scikit-learn xgboost joblib

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 9.4 MB/s eta 0:00:11
   - -------------------------------------- 2.9/101.7 MB 8.4 MB/s eta 0:00:12
   - -------------------------------------- 5.0/101.7 MB 8.2 MB/s eta 0:00:12
   --- ------------------------------------ 8.4/101.7 MB 10.6 MB/s eta 0:00:09
   ----- ---------------------------------- 12.8/101.7 MB 12.6 MB/s eta 0:00:08
   ------- -------------------------------- 18.6/101.7 MB 15.1 MB/s eta 0:00:06
   -------- ------------------------------- 21.5/101.7 MB 15.1 MB/s eta 0:00:06
   ---------- ----------------------------- 27.3/101.7 MB 16.3 MB/s eta 0:00:05
   ------------ --------------------------- 32.5/101.7 MB 17.2 MB/s eta 0:00:05
   --------------- ------------------------ 38.3/101.7 MB 18.3 MB/s eta 0:00:04
   ---------------- ----------------------- 41.9/101.7 MB 18.9 MB/s eta 0:00:04
   ----------------- ---------------------- 43.3/101.7 M

In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
ML_TOOLS_DIR = WORKDIR / 'ml_tools'
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print('WORKDIR:', WORKDIR)
print('ML_TOOLS:', ML_TOOLS_DIR)
for p in sorted((ML_TOOLS_DIR / 'data').glob('*.csv')):
    print(' - data/', p.name)

WORKDIR: c:\Users\Admin\Desktop\AI Autonomous\cursor\14일차
ML_TOOLS: c:\Users\Admin\Desktop\AI Autonomous\cursor\14일차\ml_tools
 - data/ anomaly_sensor.csv
 - data/ failure_equipment.csv


---
## 1. Day 8~11 ML 방법론 요약

| Tool | 강의 | 알고리즘 | Agent가 쓰는 상황 |
|------|------|----------|-------------------|
| `predict_failure_logistic` | 로지스틱 회귀 | 설비 고장 Yes/No + **확률** (해석·설명) |
| `predict_failure_xgboost` | XGBoost | 불균형 고장 데이터 **예측 성능** 우선 |
| `detect_anomaly_oneclass` | One-Class SVM | 정상 기준 **이상 탐지** (2D 센서) |

공통 패턴:
1. `ml_tools/`에서 **사전 학습 모델** 로드 (`models/*.joblib`)
2. `@tool` 함수가 **문자열**로 결과 반환 → Agent가 읽고 답변

---
## Step 1. ML Tool 불러오기

In [6]:
from ml_tools import ML_TOOLS, predict_failure_logistic, predict_failure_xgboost, detect_anomaly_oneclass

print('등록된 ML Tool:')
for t in ML_TOOLS:
    print(f' - {t.name}: {t.description[:60]}...')

등록된 ML Tool:
 - predict_failure_logistic: 설비 고장(Failure) 여부를 로지스틱 회귀(Day 2-2)로 예측합니다.

    Args:
     ...
 - predict_failure_xgboost: 설비 고장(Failure) 여부를 XGBoost(Day 3-2)로 예측합니다.

    Args:
     ...
 - detect_anomaly_oneclass: 2차원 센서값이 정상 범위인지 One-Class SVM(Day 4-3 / SVDD)으로 탐지합니다.

   ...


---
## ✏️ 실습 1. `@tool` 직접 만들기

**목표:** 현재 시각을 `HH:MM` 형식으로 반환하는 `get_time_hhmm` Tool을 만듭니다.

**힌트:** `from langchain_core.tools import tool` · `datetime.now().strftime('%H:%M')`

In [11]:
# ── 여기에 작성 ──
from datetime import datetime
from langchain_core.tools import tool

@tool
def get_time_hhmm() -> str:
    """현재 시각을 HH:MM 형식으로 반환합니다."""
    return datetime.now().strftime('%H:%M')

print(get_time_hhmm.invoke({}))

16:49


---
## Step 2. Tool 단독 테스트

Agent 없이 `.invoke()`로 먼저 확인합니다. (14.1의 `get_current_time.invoke`와 동일)

In [15]:
# 설비 고장 예측 — 로지스틱 회귀 (Day 2-2)
print(predict_failure_logistic.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%


In [18]:
# 설비 고장 예측 — XGBoost (Day 3-2)
print(predict_failure_xgboost.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[XGBoost] 예측: No
- 고장(Failure) 확률: 0.74%
- 정상(No) 확률: 99.26%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)


In [19]:
# 이상 탐지 — One-Class SVM (Day 4-3 / SVDD)
samples = [(1.0, 1.5, '정상에 가까운 값'), (5.0, -4.0, '이상에 가까운 값')]
for f0, f1, label in samples:
    print(f'[{label}]')
    print(detect_anomaly_oneclass.invoke({'feature_0': f0, 'feature_1': f1}))
    print('-' * 40)

[정상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------
[이상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.9268
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------


---
## 센서값 직접 바꿔 `.invoke()`

`detect_anomaly_oneclass`에 임의의 `feature_0`, `feature_1` 값을 넣어 결과를 확인하세요.

**관찰:** decision_score가 0에 가까울수록 경계선 샘플입니다.

In [20]:
# ── 여기에 작성 ──
my_f0 = 0
my_f1 = 0
print(detect_anomaly_oneclass.invoke({'feature_0': my_f0, 'feature_1': my_f1}))

[One-Class SVM] 판정: 이상
- decision_score: -0.6439
- raw_prediction: -1 (+1 정상 / -1 이상)


---
## Step 3. AgentExecutor — ML Tool 통합

14.1과 같이 `create_tool_calling_agent` + `AgentExecutor`를 씁니다.

In [36]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = ML_TOOLS
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한글로 답하세요.

온도 등 특정 변수가 입력되지 않으면 툴을 사용할 수 없다고 해줘.
'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## ✏️ 실습 3. 시스템 프롬프트 수정

Step 3의 `prompt` **system** 문자열에 아래 규칙을 **한 줄** 추가한 뒤, Agent를 다시 만들어 보세요.

> *"로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요."*

**힌트:** `agent` · `executor` 변수를 다시 할당하면 됩니다.

In [37]:
# ── 여기에 작성 ──
# (Step 3 prompt 복사 후 system 문자열에 규칙 1줄 추가)
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## Step 4. Agent 데모 질문

In [41]:
# 로지스틱 회귀 Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 90, Operator1, 이전 고장 후 90시간 — 로지스틱으로 고장 확률 알려줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 90, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 0.72%
- 정상(No) 확률: 99.28%로지스틱 회귀 예측 결과는 다음과 같습니다:

- 고장(Failure) 확률: 0.72%
- 정상(No) 확률: 99.28%

따라서, 설비 고장이 발생할 가능성은 매우 낮습니다.

> Finished chain.
로지스틱 회귀 예측 결과는 다음과 같습니다:

- 고장(Failure) 확률: 0.72%
- 정상(No) 확률: 99.28%

따라서, 설비 고장이 발생할 가능성은 매우 낮습니다.


In [42]:
# XGBoost Tool 선택 유도
r = executor.invoke({
    'input': '같은 조건으로 XGBoost 모델 예측도 해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...
온도, 습도, 작업자, 이전 고장 경과 시간이 필요합니다. 해당 정보를 제공해 주시면 XGBoost 모델을 사용하여 예측하겠습니다.

> Finished chain.
온도, 습도, 작업자, 이전 고장 경과 시간이 필요합니다. 해당 정보를 제공해 주시면 XGBoost 모델을 사용하여 예측하겠습니다.


In [43]:
# One-Class SVM 이상 탐지
r = executor.invoke({
    'input': '센서 feature_0=1.0, feature_1=1.5 일 때 정상인지 이상인지 판정해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 1.0, 'feature_1': 1.5}`


[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)판정 결과: 이상입니다. (decision_score: -0.0711)

> Finished chain.
판정 결과: 이상입니다. (decision_score: -0.0711)


---
## ✏️ 실습 4. Agent에 본인 질문 작성

**고장 예측** 또는 **이상 탐지** 중 하나를 골라, Step 4 데모와 다른 조건으로 `executor.invoke`를 직접 호출하세요.

**예시:** *"온도 80, 습도 90, Operator3, 이전 고장 10시간 — XGBoost로 예측해줘"*

In [44]:
# ── 여기에 작성 ──
my_question = '온도 80, 습도 90, Operator3, 이전 고장 10시간 — XGBoost로 예측해줘'
r = executor.invoke({'input': my_question, 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 80, 'humidity': 90, 'operator': 'Operator3', 'hours_since_previous_failure': 10}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 5.75%
- 정상(No) 확률: 94.25%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)예측 결과는 "고장 없음(No)"입니다. 고장 확률은 5.75%이며, 정상 확률은 94.25%입니다. 주요 변수로는 온도(Temperature), 이전 고장 경과 시간(Hours Since Previous Failure), 습도(Humidity)가 있습니다.

> Finished chain.
예측 결과는 "고장 없음(No)"입니다. 고장 확률은 5.75%이며, 정상 확률은 94.25%입니다. 주요 변수로는 온도(Temperature), 이전 고장 경과 시간(Hours Since Previous Failure), 습도(Humidity)가 있습니다.


---
## Step 5. 멀티턴 — `chat_history`

In [45]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('온도 70, 습도 75, Operator2, 이전 고장 120시간 — 고장 날까?'))
print('2:', session.ask('방금 답에서 고장 확률만 숫자로 다시 말해줘'))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%예측 결과, 고장(Failure) 확률은 11.09%로 낮으며, 정상(No) 확률은 88.91%입니다. 따라서 고장이 날 가능성은 낮습니다.

> Finished chain.
1: 예측 결과, 고장(Failure) 확률은 11.09%로 낮으며, 정상(No) 확률은 88.91%입니다. 따라서 고장이 날 가능성은 낮습니다.


> Entering new AgentExecutor chain...
고장 확률은 11.09%입니다.

> Finished chain.
2: 고장 확률은 11.09%입니다.


---
## ✏️ 실습 5. 멀티턴 후속 질문

`ChatSession`으로 1턴 예측 후, **2턴에서 맥락을 이어가는** 후속 질문을 직접 작성하세요.

**예시:** 1턴 고장 예측 → 2턴 *"그럼 같은 조건으로 이상 탐지도 해줘 feature_0=2, feature_1=2"*

In [46]:
# ── 여기에 작성 ──
session2 = ChatSession(executor)
q1 = '...'
q2 = '...'  # 1턴 답을 참조하는 후속 질문
print('1:', session2.ask(q1))
print('2:', session2.ask(q2))



> Entering new AgentExecutor chain...
질문이 있으시면 말씀해 주세요. 도움이 필요하신 부분에 대해 안내해 드리겠습니다.

> Finished chain.
1: 질문이 있으시면 말씀해 주세요. 도움이 필요하신 부분에 대해 안내해 드리겠습니다.


> Entering new AgentExecutor chain...
더 궁금한 점이나 도움이 필요하신 부분이 있으시면 말씀해 주세요. 언제든지 도와드리겠습니다!

> Finished chain.
2: 더 궁금한 점이나 도움이 필요하신 부분이 있으시면 말씀해 주세요. 언제든지 도와드리겠습니다!


---
## Step 6. (선택) 14.1 Agent와 합치기

RAG · 웹 검색 · ML을 한 Agent에 넣으려면 `agent_tools` 리스트에 합치면 됩니다.

### ✏️ 실습 6

1. **14.1 노트북에 정의**된 `get_current_time`, `search_pdf_tool`, `web_search`코드를 복사해 선언합니다.
2. 아래 셀에서 `agent_tools`를 채우고, `system` 프롬프트에 RAG·웹·ML Tool 용도를 각각 한 줄씩 추가하세요.
3. (선택) 합친 Agent로 *"석사 수업연한이 몇 학기야?"* 와 *"온도 70 습도 80 고장 예측"* 을 연속 질문해 보세요.

In [ ]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    
@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""

web_search = DuckDuckGoSearchResults(num_results=3)

In [49]:
from langchain_core.tools import create_retriever_tool
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)
hybrid_hits = hybrid_retriever.invoke(demo_query)
show_hits('하이브리드 BM25 + 벡터 (k=3)', hybrid_hits)

################
search_pdf_tool = create_retriever_tool(
    hybrid_retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)
# PDF → search_pdf_documents

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])

NameError: name 'vectorstore' is not defined

In [ ]:
# ── 여기에 작성 (14.1 실행 후) ──
agent_tools = [
    get_current_time,
    # search_pdf_tool,
    web_search,
    *ML_TOOLS,
]

combined_prompt = ChatPromptTemplate.from_messages([
    ('system', '''... RAG / 웹 / ML Tool 용도를 각각 적으세요 ...'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

combined_agent = create_tool_calling_agent(llm, agent_tools, combined_prompt)
combined_executor = AgentExecutor(agent=combined_agent, tools=agent_tools, verbose=True, max_iterations=8)

session = ChatSession(combined_executor)
# print('1:', session.ask('한국과 멕시코 경기 결과'))



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_results_json` with `{'query': '한국 멕시코 축구 경기 결과'}`


snippet: 1 week ago - 당시 한국은 전통적으로 멕시코를 상대로 약했던 것에 비하면 상당히 괜찮은 경기력을 보여주면서 2:2 무승부를 거두며 호평을 받은 바 있다., title: 홍명보호(성인 2기)/2026 FIFA 월드컵/멕시코전 - 나무위키, link: https://namu.wiki/w/홍명보호(성인+2기)/2026+FIFA+월드컵/멕시코전, snippet: 3 weeks ago - 한국, 멕시코에 1대 0 패배…1승 1패, 32강 진출 미확정 ... 우리나라 축구대표팀이 북중미 월드컵에서 개최국 멕시코에 져 조별리그 1승 1패로 32강 진출을 확정하지 못했습니다., title: 한국, 멕시코에 1대 0 패배…1승 1패, 32강 진출 미확정, link: https://news.kbs.co.kr/news/view.do?ncd=8590385, snippet: 3 weeks ago - 연합뉴스 · 2026 북중미 월드컵 본선 일정을 소화 중인 한국축구대표팀이 공동개최국이자 홈팀 멕시코와의 맞대결에서 아쉬운 패배를 당했다., title: 한국, 멕시코 징크스 못깼다…아쉬운 수비 실책에 0-1 패배 | 중앙일보, link: https://www.joongang.co.kr/article/25438141최근 한국과 멕시코의 축구 경기 결과는 다음과 같습니다:

- 한국은 멕시코와의 경기에서 2:2 무승부를 기록했습니다. 이 경기는 한국이 전통적으로 약했던 멕시코를 상대로 괜찮은 경기력을 보여준 것으로 평가받았습니다. [자세한 내용](https://namu.wiki/w/홍명보호(성인+2기)/2026+FIFA+월드컵/멕시코전)
  
- 그러나 이전 경기에서는 한국이 멕시코에 1:0으로 패배한 바 있습니다. 이로 인해 한국은 조별리그에서 1승 1패를 기록

---
## Step 7. `input()` 대화 루프

In [ ]:
# print('종료: exit')
# while True:
#     user_input = input('사용자: ').strip()
#     if user_input.lower() in ('exit', 'quit', '종료'):
#         break
#     if not user_input:
#         continue
#     print('AI:', session.ask(user_input))

---
## 정리

| 파일 | 역할 |
|------|------|
| `ml_tools/logistic_regression_tool.py` | Day 9 로지스틱 회귀 |
| `ml_tools/xgboost_classifier_tool.py` | Day 10 XGBoost |
| `ml_tools/oneclass_svm_tool.py` | Day 11 One-Class SVM |
| `ml_tools/models/*.joblib` | 사전 학습 모델 (자동 생성) |

> ML Tool은 **추론만** 수행합니다. 재학습이 필요하면 각 `.py`를 `python logistic_regression_tool.py`처럼 직접 실행하세요.